In [ ]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import (
    LigandReceptorNetwork,
    WeightedNetwork
)
from nichenetpy.ann_utils import prepare_ann
from nichenetpy.gene_symbol import mouse_alias_info

import anndata
import numpy as np
import os
import requests
import pickle

Here we provide some basic information on how to use NichenetPy, for more in-depth information please see the more advanced tutorials described in the [README](../README.md)

You can download the NicheNet model from Zenodo

In [2]:
filename = "nichenet_mouse.pkl"
file_path = os.path.join(
    "./tutorial_files", # set a directory to store the model here
    filename
)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/17061000/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Read the file

In [3]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())

The model is stored in a pickle file, this is a native format that allows you to store python objects. This particular pickle file contains a dictionary (key-value mapping). You can check the dictionary keys as follows...

In [4]:
list(model.keys())

['predictor', 'lr_network', 'lr_sig', 'gr']

As you can see the dictionary contains the ligand-receptor network, the weighted networks (see [model construction tutorial](model_construction.ipynb)) and the ligand activity predictor. 

In [ ]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor : LigandActivityPredictor = model["predictor"]
lr_network : LigandReceptorNetwork = model["lr_network"]
lr_sig : WeightedNetwork = model["lr_sig"]
gr : WeightedNetwork = model["gr"]

The ligand activity predictor is just a wrapper around the ligand-target matrix. 

In [ ]:
lt_matrix = predictor.ligand_target_matrix
row_names = predictor.get_genes()
col_names = predictor.get_ligands()

array([[0.00000000e+00, 0.00000000e+00, 1.31129654e-05, ...,
        0.00000000e+00, 1.04889523e-04, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.26930146e-05, ...,
        0.00000000e+00, 1.37204785e-05, 0.00000000e+00],
       [8.87290227e-05, 4.97719661e-05, 2.58190896e-04, ...,
        6.40972541e-05, 6.78965688e-05, 2.63579410e-04],
       ...,
       [3.21873789e-03, 1.38071668e-03, 4.80895951e-03, ...,
        3.56461700e-03, 3.87534885e-03, 2.74977345e-03],
       [2.70731030e-03, 1.27577896e-03, 4.05573807e-03, ...,
        2.11251246e-03, 2.79042303e-03, 2.41981489e-03],
       [3.86703101e-03, 1.67307105e-03, 5.08292706e-03, ...,
        3.06913178e-03, 5.07955575e-03, 3.50959661e-03]])

Now let's take a look at an AnnData object

In [ ]:
data_path = os.path.normpath("./tutorial_files/AnnData") # set a directory to store AnnData object here
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/15574665/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

In [9]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))

The observations contain information about the samples, to run a NicheNet analysis the samples should be cells and they should be annotated with at least the celltype and the condition. In this case we have the columns "celltype" and "aggregate". 

In [11]:
ann.obs.head()

,nGene,nUMI,orig.ident,aggregate,res.0.6,celltype,nCount_RNA,nFeature_RNA
W380370,880.0,1611.0,LN_SS,SS,1,CD8 T,1607.0,876
W380372,541.0,891.0,LN_SS,SS,0,CD4 T,885.0,536
W380374,742.0,1229.0,LN_SS,SS,0,CD4 T,1223.0,737
W380378,847.0,1546.0,LN_SS,SS,1,CD8 T,1537.0,838
W380379,839.0,1606.0,LN_SS,SS,0,CD4 T,1603.0,836


This AnnData object contains a "counts", "data" and "scale.data" layer. 

In [12]:
ann.layers.keys()

KeysView(Layers with keys: counts, data, scale.data)

Sometimes you can find data in the X attribute, this is often the case when there is only one layer present in the AnnData object. 

In [13]:
ann.X

NicheNetPy expects to find the gene names to be stored in var_names. 

In [14]:
ann.var_names

Index(['0610005C13Rik', '0610007C21Rik', '0610007L01Rik', '0610007P08Rik',
       '0610007P14Rik', '0610007P22Rik', '0610009B22Rik', '0610009D07Rik',
       '0610009L18Rik', '0610009O20Rik',
       ...
       'Zpbp2', 'Zscan18', 'Zscan2', 'Zwilch', 'mmu-mir-2134-1',
       'mmu-mir-2134-2', 'snoU109', 'snoU97', 'snoZ39', 'snoZ40'],
      dtype='object', length=13541)

In case you have trouble running a NicheNet analysis on your AnnData object you can try prepare_ann. 

In [17]:
prepare_ann(ann)

You can check how many of your genes are present in the ligand-target matrix. 

In [18]:
predictor.gene_presence(ann.var_names)

0.7487630160254043

It helps to convert gene symbol aliases. 

In [19]:
mouse_alias_info.alias_to_symbol(ann)

In [20]:
predictor.gene_presence(ann.var_names)

0.8348718706151688

The simplest way to run a NicheNet analysis is to [use the wrapper function](wrapper.ipynb). We also provide a [step-by-step tutorial](steps.ipynb) and several usecases. All notebooks are documented in the [README](../README.md). 